In [0]:
ruta = "/Volumes/workspace/ef_precios_medicamentos/datos/datos_listos.csv"
df = spark.read.csv(ruta, header=True, inferSchema=True)
df = df.withColumnRenamed(df.columns[0], "codigo_producto")  # quita el BOM del primer nombre
print("Filas:", df.count())
print("Columnas:", len(df.columns))
df.printSchema()
display(df.limit(5))

Filas: 92816
Columnas: 21
root
 |-- codigo_producto: integer (nullable = true)
 |-- nombre_comercial: string (nullable = true)
 |-- concentracion: string (nullable = true)
 |-- forma_farmaceutica: string (nullable = true)
 |-- presentacion: string (nullable = true)
 |-- registro_sanitario: string (nullable = true)
 |-- ifa: string (nullable = true)
 |-- titular: string (nullable = true)
 |-- fabricante: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- clasificacion: string (nullable = true)
 |-- n_establecimientos: integer (nullable = true)
 |-- precio_media: double (nullable = true)
 |-- precio_mediana: double (nullable = true)
 |-- precio_moda: double (nullable = true)
 |-- periodo: date (nullable = true)
 |-- mes: integer (nullable = true)
 |-- nombre_mes: string (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- anio: integer (nullable = true)
 |-- variacion_mensual_pct: double (nullable = true)



codigo_producto,nombre_comercial,concentracion,forma_farmaceutica,presentacion,registro_sanitario,ifa,titular,fabricante,pais,clasificacion,n_establecimientos,precio_media,precio_mediana,precio_moda,periodo,mes,nombre_mes,trimestre,anio,variacion_mensual_pct
165,ACTEMRA,80 MG / 4 ML,CONCENTRADO PARA SOLUCIÓN PARA PERFUSIÓN,CAJA VIAL X 4 ML,BE00552,TOCILIZUMAB,ROCHE FARMA (PERU) S.A.,"CHUGAI PHARMA MANUFACTURING CO., LTD.",JAPON,MARCA,8,650.89125,653.67,653.67,2026-01-01,1,Enero,1,2026,null
165,ACTEMRA,80 MG / 4 ML,CONCENTRADO PARA SOLUCIÓN PARA PERFUSIÓN,CAJA VIAL X 4 ML,BE00552,TOCILIZUMAB,ROCHE FARMA (PERU) S.A.,"CHUGAI PHARMA MANUFACTURING CO., LTD.",JAPON,MARCA,8,650.89125,653.67,653.67,2026-02-01,2,Febrero,1,2026,0.0
165,ACTEMRA,80 MG / 4 ML,CONCENTRADO PARA SOLUCIÓN PARA PERFUSIÓN,CAJA VIAL X 4 ML,BE00552,TOCILIZUMAB,ROCHE FARMA (PERU) S.A.,"CHUGAI PHARMA MANUFACTURING CO., LTD.",JAPON,MARCA,6,649.965,646.635,639.6,2026-03-01,3,Marzo,1,2026,-1.08
165,ACTEMRA,80 MG / 4 ML,CONCENTRADO PARA SOLUCIÓN PARA PERFUSIÓN,CAJA VIAL X 4 ML,BE00552,TOCILIZUMAB,ROCHE FARMA (PERU) S.A.,"CHUGAI PHARMA MANUFACTURING CO., LTD.",JAPON,MARCA,6,649.965,646.635,646.635,2026-04-01,4,Abril,2,2026,0.0
165,ACTEMRA,80 MG / 4 ML,CONCENTRADO PARA SOLUCIÓN PARA PERFUSIÓN,CAJA VIAL X 4 ML,BE00552,TOCILIZUMAB,ROCHE FARMA (PERU) S.A.,"CHUGAI PHARMA MANUFACTURING CO., LTD.",JAPON,MARCA,7,640.0,653.67,639.6,2026-05-01,5,Mayo,2,2026,1.09


In [0]:
from pyspark.sql import functions as F

# Estadísticas descriptivas de los precios
df.select("precio_media", "precio_mediana", "n_establecimientos").describe().show()

# Nulos por columna
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show(vertical=True)

# Duplicados exactos
print("Duplicados:", df.count() - df.dropDuplicates().count())

# Productos distintos y periodos
print("Productos:", df.select("codigo_producto").distinct().count())
df.groupBy("periodo").count().orderBy("periodo").show()

+-------+------------------+------------------+------------------+
|summary|      precio_media|    precio_mediana|n_establecimientos|
+-------+------------------+------------------+------------------+
|  count|             92816|             92816|             92816|
|   mean|118.08732341093919|117.82728522022063| 754.5790057748663|
| stddev|1241.5339876783398|1244.1980715738307|1255.6491046352576|
|    min|              0.05|              0.05|                 1|
|    max|           52397.8|           52397.8|              9920|
+-------+------------------+------------------+------------------+

-RECORD 0----------------------
 codigo_producto       | 0     
 nombre_comercial      | 0     
 concentracion         | 3073  
 forma_farmaceutica    | 0     
 presentacion          | 0     
 registro_sanitario    | 0     
 ifa                   | 0     
 titular               | 0     
 fabricante            | 0     
 pais                  | 0     
 clasificacion         | 0     
 n_estableci

In [0]:
df_limpio = (df
    .filter(F.col("precio_mediana") > 0)
    .fillna({"concentracion": "NO ESPECIFICADA"}))
print("Filas después de limpiar:", df_limpio.count())

Filas después de limpiar: 92816


In [0]:
df.write.mode("overwrite").saveAsTable("workspace.ef_precios_medicamentos.bronze_precios")
df_limpio.write.mode("overwrite").saveAsTable("workspace.ef_precios_medicamentos.silver_precios")

In [0]:
%sql
SHOW TABLES IN workspace.ef_precios_medicamentos;

database,tableName,isTemporary
ef_precios_medicamentos,bronze_precios,false
ef_precios_medicamentos,silver_precios,false


In [0]:
df_limpio.groupBy("clasificacion").count().show(truncate=False)

+-------------+-----+
|clasificacion|count|
+-------------+-----+
|MARCA        |72044|
|GENERICO     |20772|
+-------------+-----+



In [0]:
from pyspark.sql import functions as F

s = spark.table("workspace.ef_precios_medicamentos.silver_precios")
jun = s.filter(F.col("periodo") == "2026-06-01")
may = s.filter(F.col("periodo") == "2026-05-01")

# KPI 1: brecha marca vs genérico (mediana marca / mediana genérico)
med = {r["clasificacion"]: r["m"] for r in jun.groupBy("clasificacion").agg(F.median("precio_mediana").alias("m")).collect()}
kpi1 = med["MARCA"] / med["GENERICO"]

# KPI 2: % de productos con alza mensual mayor a 5%
kpi2 = jun.filter(F.col("variacion_mensual_pct").isNotNull()) \
          .agg(F.avg((F.col("variacion_mensual_pct") > 5).cast("int"))).first()[0] * 100

# KPI 3: variación mensual de la mediana de precios de genéricos (junio vs mayo)
g_jun = jun.filter(F.col("clasificacion") == "GENERICO").agg(F.median("precio_mediana")).first()[0]
g_may = may.filter(F.col("clasificacion") == "GENERICO").agg(F.median("precio_mediana")).first()[0]
kpi3 = (g_jun / g_may - 1) * 100

# KPI 4: dispersión entre laboratorios (CV del precio mediano, mismo IFA + forma + concentración, >= 3 laboratorios)
clave = ["ifa", "forma_farmaceutica", "concentracion"]
por_lab = (jun.filter(F.col("concentracion") != "NO ESPECIFICADA")
              .groupBy(clave + ["fabricante"]).agg(F.median("precio_mediana").alias("p")))
cv = (por_lab.groupBy(clave)
             .agg(F.count("*").alias("n_labs"), (F.stddev("p") / F.avg("p")).alias("cv"))
             .filter("n_labs >= 3"))
kpi4 = cv.agg(F.median("cv")).first()[0] * 100

gold = spark.createDataFrame([
    ("Brecha marca vs genérico (x)", round(kpi1, 2), "<= 2.0", "2026-06"),
    ("% productos con alza mensual > 5%", round(kpi2, 1), "<= 5", "2026-06"),
    ("Variación mensual precio mediano genéricos (%)", round(kpi3, 1), "entre -2 y +2", "2026-06"),
    ("Dispersión entre laboratorios (CV %)", round(kpi4, 1), "<= 25", "2026-06"),
], ["kpi", "valor", "objetivo", "periodo"])

gold.write.mode("overwrite").saveAsTable("workspace.ef_precios_medicamentos.gold_kpis")
display(gold)

kpi,valor,objetivo,periodo
Brecha marca vs genérico (x),2.4,<= 2.0,2026-06
% productos con alza mensual > 5%,7.1,<= 5,2026-06
Variación mensual precio mediano genéricos (%),-0.4,entre -2 y +2,2026-06
Dispersión entre laboratorios (CV %),37.5,<= 25,2026-06


In [0]:
# Precio mediano mensual: marca vs genérico
evol = (s.groupBy("periodo", "clasificacion")
          .agg(F.median("precio_mediana").alias("precio_mediano"))
          .orderBy("periodo"))
display(evol)

periodo,clasificacion,precio_mediano
2026-01-01,GENERICO,2.75
2026-01-01,MARCA,6.485
2026-02-01,MARCA,6.5
2026-02-01,GENERICO,2.75
2026-03-01,MARCA,6.57
2026-03-01,GENERICO,2.8
2026-04-01,GENERICO,2.76
2026-04-01,MARCA,6.5
2026-05-01,MARCA,6.5
2026-05-01,GENERICO,2.75


Databricks visualization. Run in Databricks to view.

In [0]:
# Número de registros por clasificación y mes
display(s.groupBy("periodo", "clasificacion").count().orderBy("periodo"))

periodo,clasificacion,count
2026-01-01,GENERICO,3408
2026-01-01,MARCA,11858
2026-02-01,MARCA,11983
2026-02-01,GENERICO,3411
2026-03-01,MARCA,12141
2026-03-01,GENERICO,3472
2026-04-01,GENERICO,3517
2026-04-01,MARCA,12018
2026-05-01,MARCA,11992
2026-05-01,GENERICO,3493


Databricks visualization. Run in Databricks to view.